In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
import sys
sys.path.append('../')
import utils.utilfunc as ut
import Jobcontrol as jc
import datetime

In [0]:
#Job Parameters
rundate = ut.get_rundate()
schema_name = 'warehouse.edw_stg'
table_name = 'dim_date_stg'
table_full_name = f"{schema_name}.{table_name}"
landing_table = "warehouse.edw_ld.dim_date_ld"
print("Job Triggered for rundate: ",rundate)


In [0]:
#Get Max Timestamp from Job_control Table
from pyspark.sql.functions import col,to_timestamp
max_timestamp = jc.get_max_timestamp(spark,schema_name,table_name)
print("Max Timestamp from Job Control Table: ",max_timestamp)

In [0]:
import datetime as dt
timeval = dt.datetime.strptime(max_timestamp,"%Y-%m-%d %H:%M:%S.%f")
print(timeval)

In [0]:
from pyspark.sql.functions import col,to_timestamp
df=spark.read.table(landing_table)
dfadd = df.where(f"insert_dt > to_timestamp('{max_timestamp}')")
dfadd = dfadd.selectExpr("to_date(date,'yyyy-MM-dd') as date","cast(day as int) as day","cast(month as int) as month","cast(year as int) as year","dayofweek","current_timestamp as insert_dt","rundate","current_timestamp as update_dt")
#dffil = df.filter(col('insert_dt') > timeval)
dfadd.show()
#dffil.show()

In [0]:
## Check Nulls by creating row_Number
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
w = Window.partitionBy("date").orderBy("insert_dt")
dfadd = dfadd.withColumn("row_num",row_number().over(w))
dfadd = dfadd.filter(col("row_num")==1).drop("row_num")
dfadd.write.mode("overwrite").saveAsTable(table_full_name)


In [0]:
#Update Job Control Table
jc.insert_log(spark,schema_name,table_name,max_timestamp,rundate)

In [0]:
#check the loading Metrics
from delta.tables import DeltaTable
dc = DeltaTable.forName(spark, table_full_name)
# Update the Delta Table History
dc.history().limit(1).select("version","operationMetrics.exectuionTimeMs","operationMetrics.numTargetRowsInserted","operationMetrics.numTargetRowsUpdated","operationMetrics.numOutputRows").display()

In [0]:
%sql
select * from warehouse.edw.job_control where schema_name = 'warehouse.edw_stg' and table_name = 'dim_date_stg'